# 03 · 벌크 vs 계면 분리 — 회절패턴 NMF (공간 분해)

계면(interface)은 **공간적 특징**이라 스캔 전체를 평균하면 벌크에 묻힙니다. 계면을 보려면 **위치별 회절
패턴을 그대로** 분해해야 합니다.

**회절패턴은 음수가 없으므로 NMF가 정확**하고 parts-based라 물리적 상 분리에 적합합니다(이게 4D-STEM
NMF의 원래 용도). 각 온도의 4D를 `X (n_position × n_detpix)`로 펼쳐 `X ≈ W · H` 로 분해:
- **H(성분)** = 상별 회절패턴 (벌크 / 계면)
- **W(loading)** = **실공간 지도** → 계면이 어디 띠로 있는지 보임
- 각 성분패턴 → `pattern_to_rdf` → **벌크 RDF · 계면 RDF**
- 온도별 **계면 성분의 양** → 계면이 사라지는 곡선

> 메모리: 4D 하나를 위치별로 NMF하면 큽니다(150×150×256×256). `DET_BIN`으로 **검출기를 비닝**해서
> `n_detpix`를 줄이세요(예: 4 → 64×64). 스캔은 그대로라 계면 지도 해상도는 유지됩니다.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DATA_ROOT = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiO"
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)

N_COMPONENTS   = 2         # 벌크 + 계면 (필요시 3으로 — 벌크/계면/기타)
DET_BIN        = 4         # 검출기 비닝(메모리). 실데이터 256→64. 합성이면 1
Q_UNIT_HINT    = "1/A"
BEAM_RADIUS_PX = 12        # (비닝 후 기준으로 자동 축소됨)
Q_PER_PX       = 0.0120    # 01의 2b 캘리브레이션 값 넣기 (비닝하면 ×DET_BIN 되어 자동 반영)
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")
print("USE_SYNTHETIC =", USE_SYNTHETIC)

## 1) 한 온도의 4D 큐브 불러오기 (+ 검출기 비닝)

실데이터: `fds.load(파일)` → `(sy,sx,qy,qx)`. 합성: 벌크(가운데 링) + 세로 계면 띠(다른 링)로 만든 데모.

In [ ]:
def make_interface_cube(scan=(60, 60), dp=(64, 64), iface_cols=(26, 34),
                        bulk_r=16.0, iface_r=22.0, seed=0):
    '''벌크(링 bulk_r) + 세로 계면 띠(iface_cols에 링 iface_r 추가) 4D 데모.'''
    rng = np.random.default_rng(seed)
    Sy, Sx = scan; H, W = dp
    yy, xx = np.mgrid[0:H, 0:W]; cx, cy = W/2, H/2
    rr = np.hypot(xx - cx, yy - cy)
    bulk  = np.exp(-(rr-bulk_r)**2/(2*3.5**2)) + 3*np.exp(-rr**2/(2*2**2))
    iface = np.exp(-(rr-iface_r)**2/(2*3.0**2)) + 0.6*np.exp(-(rr-bulk_r)**2/(2*3.5**2)) + 3*np.exp(-rr**2/(2*2**2))
    cube = np.empty((Sy, Sx, H, W), np.float32)
    for ix in range(Sx):
        pat = iface if (iface_cols[0] <= ix < iface_cols[1]) else bulk
        cube[:, ix] = pat + 0.02*rng.standard_normal((Sy, H, W))
    return np.clip(cube, 0, None)

if USE_SYNTHETIC:
    cube = fds.from_array(make_interface_cube(), q_per_px=Q_PER_PX, name="synthetic")
    DET_BIN = 1
else:
    f0 = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))[0]   # 예: 최저온
    cube = fds.load(f0, Q_UNIT_HINT)
    print("loaded", f0, cube.shape)

if DET_BIN > 1:
    cube = fds.bin_cube_detector(cube, DET_BIN)     # 검출기 축소(메모리)
qpp = (cube.calibration.q_per_px or Q_PER_PX)
print("cube for NMF:", cube.shape, "| q_per_px:", qpp)

## 1b) 먼저 가상 이미지로 스캔 확인 (계면이 보이나?)

⚠️ **주의**: 위치별 총세기는 빔커런트/두께/스캔라인 드리프트로 변합니다 → 가상이미지(ADF 등)에 **가로
줄무늬(스캔 아티팩트)** 로 나타납니다. 계면(구조 차이)을 보려면 **밝기로 정규화**해야 합니다:
`구조링 DF / 총세기`. 정규화 지도에서 세로 띠가 보이면 그게 계면입니다.

In [ ]:
mean_dp = cube.mean_dp()
(cx, cy), _ = fds.find_center(mean_dp, fds.beam_stopper_mask(mean_dp))
dp = cube.dp_shape
adf   = fds.annular_dark_field(cube, center=(cx, cy),
                               r_inner=min(dp)/6, r_outer=min(dp)/2)
struct = fds.structural_map(cube, center=(cx, cy))    # 밝기 정규화 구조맵 (계면 대비)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
ax[0].imshow(adf, cmap="viridis"); ax[0].axis("off")
ax[0].set_title("ADF (raw): horizontal stripes = scan artifact")
im = ax[1].imshow(struct, cmap="viridis"); ax[1].axis("off")
ax[1].set_title("brightness-normalized structural map (interface?)")
plt.colorbar(im, ax=ax[1], fraction=0.046); plt.tight_layout(); plt.show()

## 1c) (a) q별 대비 스캔 — 어느 구조링에서 계면이 가장 또렷한가

계면/벌크 대비는 **어느 q(링 반경)** 를 보느냐에 따라 달라집니다. 여러 링에서 구조맵을 그리고 **분리
점수**(대비/노이즈)를 비교해, 계면이 가장 선명한 q를 고릅니다. → 이 링을 `structural_map`·클러스터
특징에 쓰면 분리가 좋아집니다.

In [ ]:
rings = [(6, 12), (10, 16), (14, 20), (18, 26)]      # (r_inner, r_outer) px — 데이터 맞게 조절
fig, ax = plt.subplots(1, len(rings)+1, figsize=(3.1*(len(rings)+1), 3.3))
scores = []
for j, (ri, ro) in enumerate(rings):
    sm = fds.structural_map(cube, center=(cx, cy), r_inner=ri, r_outer=ro)
    prof = sm.mean(0)
    score = float((prof.max() - np.median(prof)) / (np.std(sm) + 1e-9))
    scores.append(score)
    ax[j].imshow(sm, cmap="viridis"); ax[j].axis("off")
    ax[j].set_title(f"r={ri}-{ro}\nscore={score:.1f}")
ax[-1].plot([f"{ri}-{ro}" for ri, ro in rings], scores, "o-")
ax[-1].set_title("separation score (higher=clearer)"); ax[-1].tick_params(axis="x", labelrotation=45)
plt.tight_layout(); plt.show()
best = rings[int(np.argmax(scores))]
print("best ring (max separation):", best)

## 1d) (b) k-means 위치 분류 — 임계값 없이 상 분할

**중요 — 왜 `feature="radial"`은 계면을 못 잡을 수 있나.** 계면은 **얇은 세로 선**(스캔의 1~2 열,
전체의 <2%)입니다. 위치별 I(q) 전체를 쓰면 그 벡터는 **중심빔·전체 밝기(두께/도즈 구배)** 가 지배하고,
2-클러스터 k-means는 항상 **가장 분산이 큰 축**(위→아래 밝기 구배)을 먼저 가릅니다. → 라벨맵이 **가로**로
나뉘고 두 상의 RDF가 거의 겹칩니다(계면이 소수라 못 이김).

**해결 — 1c에서 실제로 분리되는 신호(구조 링 대비)만 특징으로 쓴다.** `feature="structural"`은 여러 구조
링의 **밝기 상쇄 대비맵**(`structural_map`)을 특징으로 클러스터합니다. `detrend=True`면 매끈한 스캔 구배를
먼저 빼서(가우시안 배경 제거) **얇은 계면 선이 구배에 묻히지 않습니다.** 이는 threshold가 아니라, 구배를
제거한 다중 링 특징공간에서 k-means가 스스로 분할을 찾는 것입니다.

In [ ]:
# 1c에서 고른 best 링을 중심으로 몇 개 밴드 → 구조 대비 특징 (밝기 제거 + 구배 detrend)
b_in, b_out = best
c_rings = [(max(2, b_in-4), b_out-4), (b_in-2, b_out-2), (b_in, b_out), (b_in+2, b_out+2)]
labels, cpats, km = fds.cluster_cube(
    cube, n_clusters=N_COMPONENTS, center=(cx, cy),
    feature="structural", rings=c_rings, detrend=True, detrend_sigma=8.0)

fig, ax = plt.subplots(1, 3, figsize=(14, 4.3))
ax[0].imshow(labels, cmap="tab10"); ax[0].axis("off")
ax[0].set_title("k-means label map (structural feature, no threshold)")
for k in range(N_COMPONENTS):
    (ccx, ccy), _ = fds.find_center(cpats[k], fds.beam_stopper_mask(cpats[k]))
    rr = fds.pattern_to_rdf(cpats[k], qpp, CFG, center=(ccx, ccy),
                            center_beam_radius=max(1, BEAM_RADIUS_PX//max(DET_BIN, 1)))
    frac = float(np.mean(labels == k))
    ax[1].plot(rr.q, rr.Iq/np.nanmax(rr.Iq), label=f"cluster {k} ({frac*100:.0f}%)")
    ax[2].plot(rr.r, rr.Gr, label=f"cluster {k}")
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("I(q) norm"); ax[1].set_title("cluster I(q)"); ax[1].legend()
ax[2].axhline(0, color="0.8", lw=0.8); ax[2].set_xlim(0, 6)
ax[2].set_xlabel("r (Å)"); ax[2].set_ylabel("G(r)"); ax[2].set_title("cluster RDF (bulk vs interface)"); ax[2].legend()
plt.tight_layout(); plt.show()

> **얇은 선이면 픽셀 클러스터링보다 '선 위치화(line localization)'가 더 정직합니다.** 1c에서 계면이
> **또렷한 세로 선**으로 보이므로, 구조맵의 **열 평균 프로파일** 피크로 계면 열 x★를 잡고 → **x★로부터의
> 거리**로 RDF/FSDP를 비닝하는 것이 3가지 목표(벌크·계면 RDF / 영역 폭 vs T / 거리별 SRO·MRO 변화)에
> 가장 곧바로 답합니다. 이것이 **노트북 04**의 방식입니다. 위 k-means는 "두 상이 실제로 갈라지나"의
> 확인용이고, 정량 분석은 04를 쓰세요.

## 2) (참고) 영역 나눠 평균 — 임계 기반 (간단하지만 경계 왜곡 주의)

> ⚠️ threshold는 경계를 넘는 위치까지 포함해 왜곡될 수 있습니다. **위 1d) k-means를 권장**합니다.
> 아래는 빠른 비교용입니다.

In [ ]:
hi = np.percentile(struct, 85); lo = np.percentile(struct, 50)
iface_reg = struct >= hi              # 계면 (구조맵 밝은 세로 띠)
bulk_reg  = struct <= lo              # 벌크
print(f"interface positions: {int(iface_reg.sum())} | bulk positions: {int(bulk_reg.sum())}")

flat = cube._flat_patterns()          # (n_pos, qy, qx)
iface_pat = flat[iface_reg.ravel()].mean(0)
bulk_pat  = flat[bulk_reg.ravel()].mean(0)

fig, ax = plt.subplots(1, 3, figsize=(14, 4.3))
ax[0].imshow(iface_reg, cmap="Reds"); ax[0].axis("off")
ax[0].set_title("interface region (structural threshold)")
for name, pat, col in [("bulk", bulk_pat, "tab:blue"), ("interface", iface_pat, "tab:red")]:
    (ccx, ccy), _ = fds.find_center(pat, fds.beam_stopper_mask(pat))
    rr = fds.pattern_to_rdf(pat, qpp, CFG, center=(ccx, ccy),
                            center_beam_radius=max(1, BEAM_RADIUS_PX//max(DET_BIN, 1)))
    ax[1].plot(rr.q, rr.Iq/np.nanmax(rr.Iq), color=col, label=name)
    ax[2].plot(rr.r, rr.Gr, color=col, label=name)
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("I(q) norm"); ax[1].set_title("region-averaged I(q)"); ax[1].legend()
ax[2].axhline(0, color="0.8", lw=0.8); ax[2].set_xlim(0, 6)
ax[2].set_xlabel("r (Å)"); ax[2].set_ylabel("G(r)"); ax[2].set_title("bulk vs interface RDF"); ax[2].legend()
plt.tight_layout(); plt.show()

## 3) 계면이 온도에 따라 사라지는가 — 구조 대비 vs 온도

**계면 대비** = (구조맵 밝은 영역 평균 − 벌크 평균). 계면이 균질화되면 **0으로 감소**합니다. 스캔
아티팩트(밝기)는 구조맵에서 이미 제거됐으니 이 지표가 신뢰됩니다.

In [ ]:
def interface_contrast(cube_b):
    md_ = cube_b.mean_dp()
    (cxx, cyy), _ = fds.find_center(md_, fds.beam_stopper_mask(md_))
    s = fds.structural_map(cube_b, center=(cxx, cyy))
    return float(s[s >= np.percentile(s, 85)].mean() - s[s <= np.percentile(s, 50)].mean())

if USE_SYNTHETIC:
    Ts = [300, 500, 700, 900, 1100]; con = []
    for T in Ts:
        w = max(0, int(round(8*(1-(T-300)/900)))); c0 = 30 - w//2
        cu = make_interface_cube(iface_cols=(c0, c0+w) if w > 0 else (0, 0), seed=T)
        con.append(interface_contrast(fds.from_array(cu, q_per_px=Q_PER_PX)))
else:
    files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4"))); Ts, con = [], []
    for p in files:
        T = fds.coordinate_from_name(os.path.splitext(os.path.basename(p))[0])
        cb = fds.load(p, Q_UNIT_HINT)
        if DET_BIN > 1: cb = fds.bin_cube_detector(cb, DET_BIN)
        Ts.append(T); con.append(interface_contrast(cb))
        print(f"  T={T:>6.0f}K  interface contrast = {con[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(Ts, con, "o-", color="crimson")
ax.set_xlabel("temperature (K)"); ax.set_ylabel("interface structural contrast")
ax.set_title("interface disappearing vs temperature"); plt.show()

## 4) (대안) 회절패턴 NMF — 상별 회절패턴/지도

계면 회절 대비가 **충분히 크면** NMF도 벌크/계면을 나눕니다(성분=상별 회절패턴, loading=지도). 얇거나
대비가 약하면 위 2·3(영역기반)이 더 안정적입니다. **패턴별 정규화**(`normalize="sum"`)로 밝기를 제거합니다.

In [ ]:
stopper = fds.beam_stopper_mask(mean_dp)
beam = fds.disk_mask(dp, (cx, cy), max(2, BEAM_RADIUS_PX // max(DET_BIN, 1)))
mask = fds.combine_masks(stopper, beam)                  # 중심빔 + stopper 제외
res = fds.nmf_decompose(cube, n_components=N_COMPONENTS, mask=mask,
                        normalize="sum")                 # ★ 밝기 아닌 구조로 분해
print("components:", res.components.shape, " loadings(maps):", res.loadings.shape)

# 계면 성분 식별: 실공간 지도가 더 '좁게 뭉친' 성분 (localization = max/mean 비 높은 쪽)
loc = [float(np.nanmax(m) / (np.nanmean(m) + 1e-9)) for m in res.loadings]
iface_idx = int(np.argmax(loc)); bulk_idx = 1 - iface_idx if N_COMPONENTS == 2 else None
print("localization:", np.round(loc, 2), "→ 계면 성분 =", iface_idx)

fig, axes = plt.subplots(2, N_COMPONENTS, figsize=(4.2*N_COMPONENTS, 8))
for j in range(N_COMPONENTS):
    tag = "INTERFACE" if j == iface_idx else ("bulk" if j == bulk_idx else f"comp{j}")
    axes[0][j].imshow(np.log1p(res.components[j]), cmap="magma")
    axes[0][j].set_title(f"{tag}: diffraction"); axes[0][j].axis("off")
    axes[1][j].imshow(res.loadings[j], cmap="viridis")
    axes[1][j].set_title(f"{tag}: real-space map"); axes[1][j].axis("off")
plt.tight_layout(); plt.show()

### 4b) NMF 성분의 RDF

NMF 성분 **패턴(비음수)** 을 각각 `pattern_to_rdf`에 넣어 상별 G(r)를 얻습니다(위 2와 교차 검증).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for j in range(N_COMPONENTS):
    comp = res.components[j]
    (cx, cy), _ = fds.find_center(comp, fds.beam_stopper_mask(comp))
    rr = fds.pattern_to_rdf(comp, qpp, CFG, center=(cx, cy),
                            center_beam_radius=max(1, BEAM_RADIUS_PX // max(DET_BIN, 1)))
    tag = "INTERFACE" if j == iface_idx else ("bulk" if j == bulk_idx else f"comp{j}")
    ax[0].plot(rr.q, rr.Iq/np.nanmax(rr.Iq), label=tag)
    ax[1].plot(rr.r, rr.Gr, label=tag)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("I(q) norm"); ax[0].set_title("NMF component I(q)"); ax[0].legend()
ax[1].axhline(0, color="0.8", lw=0.8); ax[1].set_xlim(0, 6)
ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)"); ax[1].set_title("NMF component RDF"); ax[1].legend()
plt.tight_layout(); plt.show()

**정리** — 계면은 먼저 **밝기 정규화 구조맵(1b)** 으로 확인하고, **영역기반 평균(2)** 으로 벌크/계면 RDF를,
**구조 대비 vs 온도(3)** 로 소멸 거동을 봅니다. NMF(4)는 대비가 크면 상별 회절/RDF를 추가로 줍니다.
- **스캔 밝기 아티팩트는 반드시 정규화로 제거**(구조맵·NMF `normalize="sum"`).
- 계면 위치가 온도에 따라 **드리프트**하면 정렬이 필요(각 T의 구조맵으로 위치 재탐색하면 완화).
- `DET_BIN`: 실데이터 메모리에 맞게(4~8). 스캔(계면 지도) 해상도는 영향 없음.
- 임계 백분위(85/50)·구조링 반경은 데이터에 맞게 조절하세요.